# Configure Domain prompt
- Step1: Google Drive 마운트, 작업 폴더 생성
- Step2: CLIP 설치, CLIP 공식 템플릿 확인
- Step3: Domain Prompt 작성
- Step4: 도메인 추정용 프롬프트 임베딩 생성
- Step5: 데이터셋(PACS) 다운로드
- (추가 사항) Step6: 도메인 추정 정확도 검증, 필요시 개선 반복

### Step 1: Google Drive 마운트, 작업 폴더 생성



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

base_path = "/content/drive/MyDrive/domain_clip_PACS"

folders = [
    "prompts",
    "embeddings",
    "datasets"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("폴더 생성 완료!")

폴더 생성 완료!


### Step2: CLIP 설치, CLIP 공식 템플릿 확인

In [3]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-em7qg6jq
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-em7qg6jq
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=4de600b977f8d809a16efa64678db5492774405c03b9169bfc1e3955cc3c8fff
  Stored in directory: /tmp/pip-ephem-wheel-cache-5boffp8c/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [4]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP 로드 완료, device:", device)

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 183MiB/s]


CLIP 로드 완료, device: cuda


In [5]:
import urllib.request
import json

# CLIP 공식 템플릿 불러오기
url = "https://raw.githubusercontent.com/openai/CLIP/main/notebooks/Prompt_Engineering_for_ImageNet.ipynb"
urllib.request.urlretrieve(url, "clip_notebook.ipynb")

with open("clip_notebook.ipynb", "r") as f:
    nb = json.load(f)

# 템플릿 출력
for cell in nb['cells']:
    if 'imagenet_templates' in ''.join(cell['source']):
        print(''.join(cell['source']))
        break

imagenet_templates = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}

### Step3: Domain Prompt 작성

In [6]:
#PACS 데이터셋을 사용한 도메인 프롬프트
domain_prompts = {
    "photo": [
        "a real photo",
        "a natural photograph",
        "a realistic photographic image",
        "a color photograph",
        "a close-up photo",
    ],
    "art_painting": [
        "an art painting",
        "an oil painting",
        "a watercolor painting",
        "a hand-painted artwork",
        "a colorful painted image",
        "a non-photorealistic painting",
    ],
    "cartoon": [
        "a cartoon image",
        "a colorful cartoon illustration",
        "an animated cartoon image",
        "a comic style illustration",
        "a simple cartoon drawing",
        "a brightly colored cartoon image",
        "a vivid colorful animated illustration",
        "a saturated cartoon drawing",
    ],
    "sketch": [
        "a black and white sketch",
        "a grayscale sketch",
        "a line drawing",
        "a contour drawing",
        "an outline drawing",
        "a monochrome line drawing",
        "a black and white line art image",
        "a clean black outline drawing",
        "a sparse black line drawing on white background",
        "a sketch with only object contours",
        "a white background with thin black outlines",
        "a monochrome contour sketch",
        "a simple object outline drawing",
        "a line-art sketch with no shading",
        "a black ink outline drawing",
        "a sketch made only of edges and contours",
        "a minimal contour-only drawing",
        "a line art illustration",
        "a black and white line art drawing",
        "a minimal line-art image",
    ],
}

### Step4: 도메인 추정용 프롬프트 임베딩 생성

In [7]:
domain_embeddings = {}

for domain, prompts in domain_prompts.items():
    tokens = clip.tokenize(prompts).to(device)
    with torch.no_grad():
        embs = model.encode_text(tokens)
        embs = embs / embs.norm(dim=-1, keepdim=True)
        domain_embeddings[domain] = embs.mean(dim=0)

print("임베딩 재생성 완료!")
for domain, emb in domain_embeddings.items():
    print(f"  {domain}: shape={emb.shape}")

임베딩 재생성 완료!
  photo: shape=torch.Size([512])
  art_painting: shape=torch.Size([512])
  cartoon: shape=torch.Size([512])
  sketch: shape=torch.Size([512])


In [8]:
import os

embeddings_dir = "/content/drive/MyDrive/domain_clip/embeddings"
prompts_dir = "/content/drive/MyDrive/domain_clip/prompts"

os.makedirs(embeddings_dir, exist_ok=True)
os.makedirs(prompts_dir, exist_ok=True)

# 저장
torch.save(domain_embeddings, f"{embeddings_dir}/domain_embeddings_pacs.pt")
torch.save(domain_prompts, f"{prompts_dir}/domain_prompts_pacs.pt")

print("저장 완료!")
print(f"임베딩 → {embeddings_dir}")
print(f"프롬프트 → {prompts_dir}")

저장 완료!
임베딩 → /content/drive/MyDrive/domain_clip/embeddings
프롬프트 → /content/drive/MyDrive/domain_clip/prompts


### Step5: 데이터셋 다운로드
- Dataset download, unzip, 경로 확인
- 전체 도메인 목록 확인

In [9]:
from datasets import load_dataset

ds = load_dataset("flwrlabs/pacs")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

### (추가 사항) Step6: 도메인 추정 정확도 검증, 필요시 개선 반복

In [10]:
import torch

eval_domain_embeddings = domain_embeddings

item = ds["train"][8000]

image = item["image"].convert("RGB")
true_domain = item["domain"]

image_input = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    img_emb = model.encode_image(image_input)
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)

logit_scale = model.logit_scale.exp()

domain_names_list = list(eval_domain_embeddings.keys())
domain_stack = torch.stack([eval_domain_embeddings[d] for d in domain_names_list])

sim_tensor = img_emb @ domain_stack.t() * logit_scale
weights = sim_tensor.softmax(dim=-1).squeeze()

sims = {d: weights[i].item() for i, d in enumerate(domain_names_list)}

predicted = max(sims, key=sims.get)

print("정답 도메인:", true_domain)
print("예측 도메인:", predicted)

if predicted == true_domain:
    print("결과: 정답")
else:
    print("결과: 오답")

print("\n전체 후보별 유사도:")
for d, score in sorted(sims.items(), key=lambda x: x[1], reverse=True):
    print(f"  {d}: {score:.4f}")

정답 도메인: sketch
예측 도메인: sketch
결과: 정답

전체 후보별 유사도:
  sketch: 0.7065
  cartoon: 0.2480
  art_painting: 0.0278
  photo: 0.0177


### Step7: 최종

In [11]:
domain_prompts = {
    "photo": [
        "a real photo",
        "a natural photograph",
        "a realistic photographic image",
        "a color photograph",
        "a close-up photo",
    ],
    "cartoon": [
        "a cartoon image",
        "a colorful cartoon illustration",
        "an animated cartoon image",
        "a comic style illustration",
        "a simple cartoon drawing",
        "a brightly colored cartoon image",
        "a vivid colorful animated illustration",
        "a saturated cartoon drawing",
    ],
    "deviantart": [
        "a deviantart digital artwork",
        "a digital art illustration",
        "a highly stylized digital artwork",
    ],
    "embroidery": [
        "an embroidered artwork",
        "a stitched embroidery design",
        "an image sewn with thread",
    ],
    "graffiti": [
        "a graffiti artwork on a wall",
        "a spray-painted graffiti image",
        "street art graffiti",
    ],
    "graphic": [
        "a vector graphic design",
        "a flat graphic illustration",
        "a clean digital graphic image",
    ],
    "misc": [
        "a miscellaneous stylized image",
        "a mixed style artwork",
        "an unusual artistic image",
    ],
    "origami": [
        "an origami paper figure",
        "a folded paper object",
        "a paper craft origami image",
    ],
    "sculpture": [
        "a three-dimensional sculpture",
        "a sculpted object",
        "a carved statue artwork",
    ],
    "sketch": [
        "a black and white sketch",
        "a grayscale sketch",
        "a line drawing",
        "a contour drawing",
        "an outline drawing",
        "a monochrome line drawing",
        "a black and white line art image",
        "a clean black outline drawing",
        "a sparse black line drawing on white background",
        "a sketch with only object contours",
        "a white background with thin black outlines",
        "a monochrome contour sketch",
        "a simple object outline drawing",
        "a line-art sketch with no shading",
        "a black ink outline drawing",
        "a sketch made only of edges and contours",
        "a minimal contour-only drawing",
        "a line art illustration",
        "a black and white line art drawing",
        "a minimal line-art image",
    ],
    "sticker": [
        "a sticker illustration",
        "a colorful sticker image",
        "a cutout sticker graphic",
    ],
    "tattoo": [
        "a tattoo design",
        "a black ink tattoo drawing",
        "a tattoo style illustration",
    ],
    "toy": [
        "a plastic toy figure",
        "a plush toy",
        "a small toy model",
    ],
    "videogame": [
        "a video game rendering",
        "a 3d game character render",
        "a computer game screenshot style image",
    ],
    "drawing": [
        "a hand-drawn drawing",
        "a simple doodle drawing",
        "a colored pencil drawing",
    ],
    "rendering": [
        "a 3d rendered image",
        "a computer generated rendering",
        "a realistic digital rendering",
    ],
    "plastic": [
        "a plastic figure",
        "a shiny plastic object",
        "a molded plastic toy",
    ]
}